in `02-multi-head-attention-and-deeper-layers.ipynb`, we trained a model on the TinyStories dataset to $1.36$ loss. this is a pretty good loss, but the stories that the model tells were still slightly off and were largely uncreative. in this notebook, i look to do some post-training using local llms, by generating many example stories given some starter prompt. given the generated stories $\mathbf{s} = \{s_1, s_2, \dots, s_M \}$, the llm chooses the best story; we can then do *direct preference optimisation (DPO)* which nudges the weights to be more likely to produce to preferred output.

In [103]:
import sys
import torch
import torch.nn as nn
import pandas as pd
import os
import json
import copy
import random

from ollama import chat
from pydantic import BaseModel
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer
from pathlib import Path
from typing import List


# import local libs
cwd = Path.cwd()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "src").is_dir())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.models.generate import generate, clean
from src.models.transformer_lm import GPTNeo

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

first, we need to use the validation set to create a bunch of example prompts. from these, we generate four outputs each. we take the first sentence as the prompt, delimiting by punctuation (`. ; !; ?`).

In [105]:
# lost test data
data_path = 'C:/data/tinystories/data'
data_files = os.listdir(data_path)
valid_files = [f for f in data_files if 'validation' in f]
tst_data = pd.concat([pd.read_parquet(f'{data_path}/{v}') for v in valid_files])

# pull in model, weights and tokeniser
model = GPTNeo(4, 10_000, 768, 512, [3024] * 4, [8] * 4, dropout=[0] * 4).to(device)
state_dict = torch.load('C:/dev/storyteller/weights/20260926_3024_4x_8_4x_0_4x_transformer.pt')
model.load_state_dict(state_dict);
tokeniser = AutoTokenizer.from_pretrained('C:/dev/storyteller/weights/gpt-neo-10k-tokeniser');

In [42]:
def extract_prompt(text: str, delimiters: List[str] = ['.', '?', '!']) -> str:
    indices = [text.find(c) for c in delimiters if text.find(c) != -1]

    assert len(indices) != 0

    first_idx = min(indices)
    return text[:first_idx+1]

tst_data['prompt'] = tst_data['text'].apply(lambda x: extract_prompt(x))
# first story is a bit weird
tst_data = tst_data[1:]

# now sample 2500 prompts from the test data
dpo_tr = tst_data.sample(2_500).reset_index(drop=True)

now that we have the prompts (the beginning sentences) generated, we can generate 4 different stories each given the prompts at temperature 1, to allow for some variation between the stories. we separate out the prompt (such that the llm does not reward the completions including the prompt) and the completions.

In [52]:
import json

output_path = 'C:/data/tinystories/dpo_stories.jsonl'
prompts_and_stories = []

with open(output_path, 'w') as f:
    for idx, row in dpo_tr.iterrows():
        print(f'prompt :: {row.prompt} :: ({round((idx / len(dpo_tr)* 100), 3)}%)')

        # generate four stories per prompt, and create an indexed record
        stories = [generate(row.prompt, model, tokeniser, device) for _ in range(4)]
        record = {
            'prompt_id': len(prompts_and_stories),
            'prompt': row.prompt,
            'completions': [
                {'completion_id': story_idx, 'completion': clean(story[len(row.prompt):])}
                for story_idx, story in enumerate(stories)
            ]
        }

        prompts_and_stories.append(record)
        f.write(json.dumps(record) + '\n')
        f.flush()

prompt :: Anna was a girl who liked to play in the park. :: (0.0%)
prompt :: Once upon a time, there lived a king in a far away country. :: (0.04%)
prompt :: Bob was walking in the park one day. :: (0.08%)
prompt :: Once upon a time, there lived two friends called Lucy and Jack. :: (0.12%)
prompt :: Mama and Jack went to the park. :: (0.16%)
prompt :: Once upon a time there was a little girl called May. :: (0.2%)
prompt :: Lily and Ben were playing in the park. :: (0.24%)
prompt :: One day, Little Jen was playing with her toys at home. :: (0.28%)
prompt :: Once there was a rabbit named Radish. :: (0.32%)
prompt :: Once there was a boy called George. :: (0.36%)
prompt :: Once upon a time there was a boy named Jack. :: (0.4%)
prompt :: Once upon a time, there was a long journey over a big hill. :: (0.44%)
prompt :: Once upon a time, in a small town, there was a unique tree. :: (0.48%)
prompt :: Once upon a time, there was a bear that wanted to find something good. :: (0.52%)
prompt :: On

now that we have example prompts and their completions from the four layer transformer model, we can begin to construct a system prompt which focuses on the characteristics of storytelling that we care about, and also appropiately deals with classifying the desired story and it's completion id (the primary key is `(prompt_id, completion_id)` for the desired completion id). we need to focus on a few things:
- reduce bias in things we don't really care about, like story length
- focus on proper *grammar* and attention to subjects in the story (something the model isn't 100% good at yet)
- focus on story *creativity*

In [72]:
class Preference(BaseModel):
    completion_id: int

def build_prompt(object: dict) -> str:
    # copy then shuffle so position doesn't line up with completion_id
    completions = object['completions'][:]
    random.shuffle(completions)
    completions = " ".join(f"[Completion {c['completion_id']}] {c['completion']}" for c in completions)

    return (
        "You are judging completions of children's stories for ages 3-4, where simple words and plots are expected and are not a weakness. "
        "Choose the best of the four completions that continue the story beginning, judging in this order of importance: "
        "1) coherence: keeps the same characters, has no contradictions or repetition, and stays on the story's topic; "
        "2) grammar: correct sentences and a proper ending; "
        "3) creativity: something interesting or warm happens that a young child would enjoy. "
        "A plain coherent story beats a creative one that falls apart, and length should not affect your choice. "
        "Respond with the completion_id of the best completion. "
        f"[Story beginning] {object['prompt']} {completions}"
    )

def preference(object: dict, model: str = 'qwen3.5:4b') -> dict:
    prompt = build_prompt(object)
    response = chat(model, messages=[{'role': 'user', 'content': prompt}], think=False, format=Preference.model_json_schema(), options={ 'temperature': 0 })

    choice = Preference.model_validate_json(response.message.content)
    return {'prompt_id': object['prompt_id'], 'completion_id': choice.completion_id}

initial runs showed that the model would be biased towards the first completion, and so we shuffle around the completions.

In [73]:
preferences_path = 'C:/data/tinystories/dpo_preferences.jsonl'
with open(preferences_path, 'w') as f, ThreadPoolExecutor(max_workers=12) as pool:
    for idx, result in enumerate(pool.map(preference, prompts_and_stories)):
        print(f'judged :: {result} :: ({round((idx + 1) / len(prompts_and_stories) * 100, 3)}%)')
        f.write(json.dumps(result) + '\n')
        f.flush()

judged :: {'prompt_id': 0, 'completion_id': 0} :: (0.04%)
judged :: {'prompt_id': 1, 'completion_id': 0} :: (0.08%)
judged :: {'prompt_id': 2, 'completion_id': 0} :: (0.12%)
judged :: {'prompt_id': 3, 'completion_id': 0} :: (0.16%)
judged :: {'prompt_id': 4, 'completion_id': 0} :: (0.2%)
judged :: {'prompt_id': 5, 'completion_id': 2} :: (0.24%)
judged :: {'prompt_id': 6, 'completion_id': 2} :: (0.28%)
judged :: {'prompt_id': 7, 'completion_id': 2} :: (0.32%)
judged :: {'prompt_id': 8, 'completion_id': 0} :: (0.36%)
judged :: {'prompt_id': 9, 'completion_id': 2} :: (0.4%)
judged :: {'prompt_id': 10, 'completion_id': 0} :: (0.44%)
judged :: {'prompt_id': 11, 'completion_id': 2} :: (0.48%)
judged :: {'prompt_id': 12, 'completion_id': 1} :: (0.52%)
judged :: {'prompt_id': 13, 'completion_id': 0} :: (0.56%)
judged :: {'prompt_id': 14, 'completion_id': 0} :: (0.6%)
judged :: {'prompt_id': 15, 'completion_id': 3} :: (0.64%)
judged :: {'prompt_id': 16, 'completion_id': 3} :: (0.68%)
judged :: 

now we have preferred completions from Qwen 3.5, we go through the theory behind direct prefernce optimisation. first, the models probability of writing some sequence of tokens $y = (y_1, y_2, \dots, y_N)$ is the product of the probability of the tokens produced given prior tokens; or in log space a sum.

$$ 
\log \pi_\theta(y | x) = \sum_{t=1}^m \log \pi_\theta(y_t | x, y_{<t})
$$

where $x$ is some prompt. these probabilities can be derived from a softmax. how *much* a model $x$ and it's evolution $e$ differ for the production of some sequence $y$ can be defined by
$$
h(y) = \log \pi_e(y) - \log \pi_x(y)
$$
such that the evolution $e$ is rewarded for being more probable to create this completion. we define the margin $u$ as $u = \beta(h(y_{accepted}) - h(y_{rejected}))$. we can group at the model level to find
$$
u = \beta([\log\pi_e(y_{accepted}) - \log\pi_e(y_{rejected})] - [\log\pi_x(y_{accepted}) - \log\pi_x(y_{rejected})])
$$
and the loss is defined by the log sigmoid of the margin
$$
L = -\log\sigma(u)
$$
thus, in a minimisation, the evolution is "encouraged" to favour the accepted completion over the rejected one.

In [99]:
preferences = []
with open(preferences_path, 'r') as f:
    for line in f:
        preferences.append(json.loads(line))

dpo_pairs = []
for idx, ps in enumerate(prompts_and_stories):
    if ps['prompt_id'] == preferences[idx]['prompt_id']:
        preferred_completion = preferences[idx]['completion_id']
        chosen = [c['completion'] for c in ps['completions'] if c['completion_id'] == preferences[idx]['completion_id']]
        if len(chosen) == 0: continue
        chosen = chosen[0]

        dpo_pairs.extend(
            {'prompt': ps['prompt'], 'chosen': chosen, 'rejected': c['completion']}
            for c in ps['completions'] if c['completion_id'] != preferences[idx]['completion_id']
        )

In [96]:
def tokenise_pair(prompt: str, completion: str, tokeniser: AutoTokenizer, context_length: int = 512) -> tuple[list[int], int]:
    prompt_tokens = tokeniser.encode(prompt)
    completion_tokens = tokeniser.encode(' ' + completion) + [tokeniser.eos_token_id]
    tokens = (prompt_tokens + completion_tokens)[:context_length]
    return tokens, len(prompt_tokens)

def completion_prob(model: nn.Module, tokens: list[int], prompt_length: int, device) -> torch.Tensor:
    tokens = torch.tensor(tokens, dtype=torch.long, device=device)
    # this produces the logits for 
    model_logits = model(tokens.unsqueeze(0)).squeeze(0)[:-1]
    log_probs = torch.log_softmax(model_logits, dim=-1)

    # for each output, get the probability it outputs for the actual token
    # for example 'the brown fox jumped over the lazy fox'
    # the first token is skipped, and has p(brown | the), p(fox | the brown), etc.
    targets = tokens[1:]
    token_log_probs = log_probs[torch.arange(targets.size(0)), targets]

    return token_log_probs[prompt_length - 1:].sum()

we now post-train the model with DPO at a learning rate of `5e-6`, which is much lower than our initial training phase. this is due to the fact that this is *already* a trained model, and setting a higher learning rate can cause significant overfitting or overshooting given the DPO objective.

In [ ]:
beta, lr, batch_size, epochs = 0.1, 5e-6, 16, 4
reference = copy.deepcopy(model).eval()
for p in reference.parameters(): p.requires_grad_(False)

model.train()
optimiser = torch.optim.AdamW(model.parameters(), lr=lr)

for epoch in range(epochs):
    random.shuffle(dpo_pairs)
    running_loss, running_correct = 0, 0

    for idx, pair in enumerate(dpo_pairs):
        chosen_tokens, prompt_length = tokenise_pair(pair['prompt'], pair['chosen'], tokeniser)
        rejected_tokens, _ = tokenise_pair(pair['prompt'], pair['rejected'], tokeniser)

        # if the policy margin > reference margin, then the model has become more likely to produce the intended completion,
        # which we reward with some sensitivity parameter beta.
        with torch.no_grad():
            reference_margin = completion_prob(reference, chosen_tokens, prompt_length, device) - completion_prob(reference, rejected_tokens, prompt_length, device)
        policy_margin = completion_prob(model, chosen_tokens, prompt_length, device) - completion_prob(model, rejected_tokens, prompt_length, device)

        # u = beta * (h(chosen) - h(rejected)), rearranged as policy gap minus reference gap
        margin = beta * (policy_margin - reference_margin)
        loss = -torch.nn.functional.logsigmoid(margin)
        (loss / batch_size).backward()

        if (idx + 1) % batch_size == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimiser.step()
            optimiser.zero_grad()

        running_loss += loss.item()
        running_correct += int(margin.item() > 0)

        if (idx + 1) % 200 == 0:
            print(f'epoch :: {epoch} :: step {idx + 1}/{len(dpo_pairs)} :: loss {round(running_loss / 200, 4)} :: accuracy {round(running_correct / 200, 3)}')
            running_loss, running_correct = 0, 0

model.eval()
torch.save(model.state_dict(), 'C:/dev/storyteller/weights/20260927_3024_4x_8_4x_0_4x_transformer_dpo.pt')

epoch :: 0 :: step 200/7482 :: loss 0.6915 :: accuracy 0.57
epoch :: 0 :: step 400/7482 :: loss 0.6901 :: accuracy 0.59
epoch :: 0 :: step 600/7482 :: loss 0.6853 :: accuracy 0.66
epoch :: 0 :: step 800/7482 :: loss 0.6829 :: accuracy 0.655
epoch :: 0 :: step 1000/7482 :: loss 0.6758 :: accuracy 0.755
epoch :: 0 :: step 1200/7482 :: loss 0.6737 :: accuracy 0.725
epoch :: 0 :: step 1400/7482 :: loss 0.6733 :: accuracy 0.68
epoch :: 0 :: step 1600/7482 :: loss 0.6645 :: accuracy 0.73
epoch :: 0 :: step 1800/7482 :: loss 0.6695 :: accuracy 0.72
epoch :: 0 :: step 2000/7482 :: loss 0.6609 :: accuracy 0.725
epoch :: 0 :: step 2200/7482 :: loss 0.6635 :: accuracy 0.735
epoch :: 0 :: step 2400/7482 :: loss 0.6603 :: accuracy 0.735
epoch :: 0 :: step 2600/7482 :: loss 0.6495 :: accuracy 0.74
epoch :: 0 :: step 2800/7482 :: loss 0.6576 :: accuracy 0.72
epoch :: 0 :: step 3000/7482 :: loss 0.6549 :: accuracy 0.74
epoch :: 0 :: step 3200/7482 :: loss 0.6472 :: accuracy 0.76
epoch :: 0 :: step 340

: 

our next step is to evaluate whether this post-trained model generalises for Qwen's preferences (as well as ours, which is more of a qualitative preference) by checking a fresh batch of prompts/completions. this is completed in the next notebook.